# 🌍 Building Change Detection in Satellite Imagery Using Deep Learning

**Project Goal:** Detect building changes (construction/demolition) from bi-temporal satellite images using a Siamese U-Net deep learning model.

**Dataset:** LEVIR-CD (Large-scale Remote Sensing Change Detection Dataset)
- 637 pairs of high-resolution (0.5m/pixel) Google Earth images
- 1024×1024 pixels each, covering 20+ years of urban growth
- Binary change masks (building change vs. no change)

**Why this project?**
This directly aligns with Professor Jonathan Li's GIM Lab research at the University of Waterloo, specifically:
- **Urban monitoring using remote sensing** — detecting building changes over time
- **Deep learning for geospatial data** — applying CNNs to satellite imagery
- **Time-series earth observations** — bi-temporal analysis for urban dynamics

---

## Step 1: Environment Setup & Dependencies

In [ ]:
# Install required packages
!pip install -q segmentation-models-pytorch albumentations huggingface_hub datasets

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import random
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 2: Download LEVIR-CD Dataset from Hugging Face

We use the LEVIR-CD dataset, which is the most widely used benchmark for building change detection.
The dataset is available on Hugging Face and contains pre- and post-change image pairs with ground truth masks.

In [ ]:
from huggingface_hub import login

# Login with your Hugging Face token
login(token='YOUR_HF_TOKEN')  # Replace with your Hugging Face token

In [ ]:
from datasets import load_dataset

# Load the LEVIR-CD dataset from Hugging Face
# This dataset contains 256x256 pre-cropped patches from the original LEVIR-CD
# Columns: imageA (before), imageB (after), label (change mask)
print("Downloading LEVIR-CD dataset from Hugging Face...")
print("This may take a few minutes on first run.")

dataset = load_dataset("ericyu/LEVIRCD_Cropped256")
print(f"\nDataset loaded successfully!")
print(f"Splits: {list(dataset.keys())}")
for split in dataset:
    print(f"  {split}: {len(dataset[split])} samples")

In [ ]:
# Examine the dataset structure
sample = dataset['train'][0]
print("Dataset columns:", dataset['train'].column_names)
print(f"\nSample keys: {list(sample.keys())}")
for key, value in sample.items():
    if hasattr(value, 'size'):
        print(f"  {key}: type={type(value).__name__}, size={value.size}, mode={value.mode}")
    else:
        print(f"  {key}: type={type(value).__name__}, value={value}")

## Step 3: Data Exploration & Visualization

Let's visualize some examples to understand the data better. Each sample has:
- **Image A (before):** The earlier satellite image
- **Image B (after):** The later satellite image  
- **Change mask:** Binary mask where white = building change

In [ ]:
def visualize_samples(dataset, split='train', num_samples=5, seed=42):
    """Visualize random samples from the dataset."""
    random.seed(seed)
    indices = random.sample(range(len(dataset[split])), min(num_samples, len(dataset[split])))
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    if num_samples == 1:
        axes = axes[np.newaxis, :]
    
    for i, idx in enumerate(indices):
        sample = dataset[split][idx]
        
        # Get images (columns: imageA, imageB, label)
        img_a = np.array(sample['imageA'])
        img_b = np.array(sample['imageB'])
        mask = np.array(sample['label'])
        
        # Normalize mask to 0-1 if needed
        if mask.max() > 1:
            mask = (mask > 127).astype(np.uint8)
        
        axes[i, 0].imshow(img_a)
        axes[i, 0].set_title(f'Image A (Before) - #{idx}', fontsize=11)
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(img_b)
        axes[i, 1].set_title(f'Image B (After) - #{idx}', fontsize=11)
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(mask, cmap='hot')
        change_pct = mask.sum() / mask.size * 100
        axes[i, 2].set_title(f'Change Mask ({change_pct:.1f}% changed)', fontsize=11)
        axes[i, 2].axis('off')
    
    plt.suptitle('LEVIR-CD Dataset Samples: Building Change Detection', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('sample_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_samples(dataset, split='train', num_samples=5)

In [ ]:
# Dataset statistics
def compute_dataset_stats(dataset, split='train', max_samples=500):
    """Compute basic statistics about the change masks."""
    change_percentages = []
    has_change = 0
    n = min(max_samples, len(dataset[split]))
    
    for i in tqdm(range(n), desc=f'Analyzing {split} set'):
        sample = dataset[split][i]
        mask = np.array(sample['label'])
        if mask.max() > 1:
            mask = (mask > 127).astype(np.uint8)
        
        pct = mask.sum() / mask.size * 100
        change_percentages.append(pct)
        if pct > 0:
            has_change += 1
    
    print(f"\n--- {split.upper()} Set Statistics (first {n} samples) ---")
    print(f"Samples with change: {has_change}/{n} ({has_change/n*100:.1f}%)")
    print(f"Mean change area: {np.mean(change_percentages):.2f}%")
    print(f"Median change area: {np.median(change_percentages):.2f}%")
    print(f"Max change area: {np.max(change_percentages):.2f}%")
    
    # Plot distribution
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(change_percentages, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_xlabel('Change Area (%)', fontsize=12)
    ax.set_ylabel('Number of Samples', fontsize=12)
    ax.set_title(f'Distribution of Change Area in {split.capitalize()} Set', fontsize=13)
    ax.axvline(np.mean(change_percentages), color='red', linestyle='--', label=f'Mean: {np.mean(change_percentages):.2f}%')
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{split}_change_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return change_percentages

train_stats = compute_dataset_stats(dataset, 'train')

## Step 4: Data Pipeline (PyTorch Dataset & Augmentations)

We create a custom PyTorch Dataset that:
1. Loads image pairs (before/after) from the HuggingFace dataset
2. Applies consistent augmentations to both images AND the mask
3. Concatenates the two images as a 6-channel input (RGB + RGB)

In [ ]:
class LEVIRCDDataset(Dataset):
    """PyTorch Dataset for LEVIR-CD change detection."""
    
    def __init__(self, hf_dataset, split='train', img_size=256, augment=False):
        self.data = hf_dataset[split]
        self.img_size = img_size
        self.augment = augment
        
        # Column names for ericyu/LEVIRCD_Cropped256
        self.col_a = 'imageA'
        self.col_b = 'imageB'
        self.col_mask = 'label'
        
        # Define augmentations
        if augment:
            self.transform = A.Compose([
                A.Resize(img_size, img_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
                A.GaussNoise(var_limit=(10, 50), p=0.2),
            ],
            additional_targets={'image_b': 'image'}
            )
        else:
            self.transform = A.Compose([
                A.Resize(img_size, img_size),
            ],
            additional_targets={'image_b': 'image'}
            )
        
        # Normalization (ImageNet stats)
        self.normalize = A.Compose([
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data[idx]
        
        # Load images
        img_a = np.array(sample[self.col_a].convert('RGB'))
        img_b = np.array(sample[self.col_b].convert('RGB'))
        mask = np.array(sample[self.col_mask].convert('L'))
        
        # Binarize mask
        mask = (mask > 127).astype(np.float32)
        
        # Apply spatial augmentations (same transform to both images and mask)
        augmented = self.transform(image=img_a, image_b=img_b, mask=mask)
        img_a = augmented['image']
        img_b = augmented['image_b']
        mask = augmented['mask']
        
        # Normalize and convert to tensor
        norm_a = self.normalize(image=img_a)
        norm_b = self.normalize(image=img_b)
        
        img_a_tensor = norm_a['image']  # [3, H, W]
        img_b_tensor = norm_b['image']  # [3, H, W]
        
        # Concatenate as 6-channel input
        img_concat = torch.cat([img_a_tensor, img_b_tensor], dim=0)  # [6, H, W]
        
        # Mask to tensor
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()  # [1, H, W]
        
        return img_concat, mask_tensor

In [ ]:
# Configuration
IMG_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 0  # Use 0 on Kaggle/Colab to avoid multiprocessing worker cleanup errors

# Create datasets
train_dataset = LEVIRCDDataset(dataset, split='train', img_size=IMG_SIZE, augment=True)
val_dataset = LEVIRCDDataset(dataset, split='val', img_size=IMG_SIZE, augment=False)
test_dataset = LEVIRCDDataset(dataset, split='test', img_size=IMG_SIZE, augment=False)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                         num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} samples, {len(val_loader)} batches")
print(f"Test:  {len(test_dataset)} samples, {len(test_loader)} batches")

# Verify a batch
batch_imgs, batch_masks = next(iter(train_loader))
print(f"\nBatch images shape: {batch_imgs.shape}  (batch, 6-channels, H, W)")
print(f"Batch masks shape:  {batch_masks.shape}  (batch, 1, H, W)")

## Step 5: Model Architecture — Siamese U-Net for Change Detection

We use a **U-Net with a ResNet-34 encoder** from `segmentation_models_pytorch`.

**Architecture Strategy:**
- Input: 6-channel concatenated image (before + after, each RGB)
- Encoder: ResNet-34 (pretrained on ImageNet, first conv modified for 6 channels)
- Decoder: U-Net decoder with skip connections
- Output: Binary change mask (1 channel, sigmoid activation)

This is a simple but effective approach used in many change detection papers.

In [ ]:
def build_change_detection_model(encoder_name='resnet34', in_channels=6, classes=1):
    """Build a U-Net model for change detection with 6-channel input."""
    model = smp.Unet(
        encoder_name=encoder_name,
        encoder_weights='imagenet',
        in_channels=in_channels,   # 6 channels: concat of before+after RGB
        classes=classes,            # Binary: change / no-change
        activation=None,            # We'll apply sigmoid in the loss function
    )
    return model

model = build_change_detection_model()
model = model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: U-Net with ResNet-34 encoder")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Step 6: Loss Function & Metrics

For change detection (highly imbalanced - most pixels are "no change"), we use:
- **Combined Loss:** BCE + Dice Loss (handles class imbalance well)
- **Metrics:** F1-Score (most important for change detection), Precision, Recall, IoU

In [ ]:
class BCEDiceLoss(nn.Module):
    """Combined Binary Cross-Entropy and Dice Loss."""
    def __init__(self, bce_weight=0.5, dice_weight=0.5, smooth=1e-6):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()
    
    def forward(self, pred, target):
        # BCE loss
        bce_loss = self.bce(pred, target)
        
        # Dice loss
        pred_sigmoid = torch.sigmoid(pred)
        pred_flat = pred_sigmoid.view(-1)
        target_flat = target.view(-1)
        intersection = (pred_flat * target_flat).sum()
        dice = (2. * intersection + self.smooth) / (pred_flat.sum() + target_flat.sum() + self.smooth)
        dice_loss = 1 - dice
        
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss


class ChangeDetectionMetrics:
    """Compute F1, Precision, Recall, and IoU for change detection."""
    def __init__(self, threshold=0.5):
        self.threshold = threshold
        self.reset()
    
    def reset(self):
        self.tp = 0
        self.fp = 0
        self.fn = 0
        self.tn = 0
    
    def update(self, pred, target):
        pred_binary = (torch.sigmoid(pred) > self.threshold).float()
        target_binary = target.float()
        
        self.tp += ((pred_binary == 1) & (target_binary == 1)).sum().item()
        self.fp += ((pred_binary == 1) & (target_binary == 0)).sum().item()
        self.fn += ((pred_binary == 0) & (target_binary == 1)).sum().item()
        self.tn += ((pred_binary == 0) & (target_binary == 0)).sum().item()
    
    def compute(self):
        precision = self.tp / (self.tp + self.fp + 1e-8)
        recall = self.tp / (self.tp + self.fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        iou = self.tp / (self.tp + self.fp + self.fn + 1e-8)
        overall_acc = (self.tp + self.tn) / (self.tp + self.tn + self.fp + self.fn + 1e-8)
        return {
            'F1': f1,
            'Precision': precision,
            'Recall': recall,
            'IoU': iou,
            'OA': overall_acc
        }

## Step 7: Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    model.train()
    running_loss = 0.0
    metrics = ChangeDetectionMetrics()
    
    pbar = tqdm(loader, desc=f'Epoch {epoch} [Train]', leave=False)
    for imgs, masks in pbar:
        imgs = imgs.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        metrics.update(outputs.detach(), masks)
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = running_loss / len(loader)
    scores = metrics.compute()
    return avg_loss, scores


@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch, phase='Val'):
    model.eval()
    running_loss = 0.0
    metrics = ChangeDetectionMetrics()
    
    pbar = tqdm(loader, desc=f'Epoch {epoch} [{phase}]', leave=False)
    for imgs, masks in pbar:
        imgs = imgs.to(device)
        masks = masks.to(device)
        
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        
        running_loss += loss.item()
        metrics.update(outputs, masks)
    
    avg_loss = running_loss / len(loader)
    scores = metrics.compute()
    return avg_loss, scores

In [ ]:
# Training configuration
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

criterion = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

# Training history
history = {
    'train_loss': [], 'val_loss': [],
    'train_f1': [], 'val_f1': [],
    'train_iou': [], 'val_iou': [],
    'lr': []
}

best_val_f1 = 0.0
best_epoch = 0

print(f"{'='*70}")
print(f"Training Configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Optimizer: AdamW (weight_decay={WEIGHT_DECAY})")
print(f"  Scheduler: CosineAnnealingLR")
print(f"  Loss: BCE + Dice")
print(f"{'='*70}")

In [ ]:
# Main training loop
for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    train_loss, train_scores = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, epoch)
    
    # Validate
    val_loss, val_scores = evaluate(model, val_loader, criterion, DEVICE, epoch, 'Val')
    
    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Log history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_f1'].append(train_scores['F1'])
    history['val_f1'].append(val_scores['F1'])
    history['train_iou'].append(train_scores['IoU'])
    history['val_iou'].append(val_scores['IoU'])
    history['lr'].append(current_lr)
    
    # Save best model
    if val_scores['F1'] > best_val_f1:
        best_val_f1 = val_scores['F1']
        best_epoch = epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': best_val_f1,
            'val_scores': val_scores,
        }, 'best_model.pth')
        marker = ' ⭐ BEST'
    else:
        marker = ''
    
    # Print epoch summary
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f}, F1: {train_scores['F1']:.4f} | "
          f"Val Loss: {val_loss:.4f}, F1: {val_scores['F1']:.4f}, IoU: {val_scores['IoU']:.4f}{marker}")

print(f"\n{'='*70}")
print(f"Training complete! Best val F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"{'='*70}")

## Step 8: Training Curves Visualization

In [ ]:
def plot_training_history(history):
    """Plot training curves."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Validation', linewidth=2)
    axes[0].set_title('Loss (BCE + Dice)', fontsize=13)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # F1 Score
    axes[1].plot(epochs, history['train_f1'], 'b-', label='Train', linewidth=2)
    axes[1].plot(epochs, history['val_f1'], 'r-', label='Validation', linewidth=2)
    best_idx = np.argmax(history['val_f1'])
    axes[1].axvline(best_idx + 1, color='green', linestyle='--', alpha=0.7, label=f'Best: {history["val_f1"][best_idx]:.4f}')
    axes[1].set_title('F1 Score', fontsize=13)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    # Learning Rate
    axes[2].plot(epochs, history['lr'], 'g-', linewidth=2)
    axes[2].set_title('Learning Rate Schedule', fontsize=13)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('LR')
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle('Training Progress: Building Change Detection (LEVIR-CD)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_history(history)

## Step 9: Test Set Evaluation

In [ ]:
# Load best model and evaluate on test set
checkpoint = torch.load('best_model.pth', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']} (Val F1: {checkpoint['val_f1']:.4f})")

test_loss, test_scores = evaluate(model, test_loader, criterion, DEVICE, 0, 'Test')

print(f"\n{'='*50}")
print(f"       TEST SET RESULTS")
print(f"{'='*50}")
print(f"  Loss:      {test_loss:.4f}")
print(f"  F1 Score:  {test_scores['F1']:.4f}")
print(f"  IoU:       {test_scores['IoU']:.4f}")
print(f"  Precision: {test_scores['Precision']:.4f}")
print(f"  Recall:    {test_scores['Recall']:.4f}")
print(f"  OA:        {test_scores['OA']:.4f}")
print(f"{'='*50}")

## Step 10: Qualitative Results — Prediction Visualization

In [ ]:
def denormalize(tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """Denormalize a tensor image."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)


@torch.no_grad()
def visualize_predictions(model, dataset, device, num_samples=8, threshold=0.5):
    """Visualize model predictions vs ground truth."""
    model.eval()
    indices = random.sample(range(len(dataset)), num_samples)
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    
    for i, idx in enumerate(indices):
        imgs, mask = dataset[idx]
        imgs_batch = imgs.unsqueeze(0).to(device)
        
        # Predict
        pred = torch.sigmoid(model(imgs_batch)).cpu().squeeze()
        pred_binary = (pred > threshold).float()
        
        # Denormalize images
        img_a = denormalize(imgs[:3])
        img_b = denormalize(imgs[3:])
        
        # Plot
        axes[i, 0].imshow(img_a.permute(1, 2, 0).numpy())
        axes[i, 0].set_title('Before', fontsize=11)
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(img_b.permute(1, 2, 0).numpy())
        axes[i, 1].set_title('After', fontsize=11)
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(mask.squeeze().numpy(), cmap='hot')
        axes[i, 2].set_title('Ground Truth', fontsize=11)
        axes[i, 2].axis('off')
        
        axes[i, 3].imshow(pred_binary.numpy(), cmap='hot')
        axes[i, 3].set_title('Prediction', fontsize=11)
        axes[i, 3].axis('off')
    
    plt.suptitle('Change Detection Results: Before | After | Ground Truth | Prediction', 
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('prediction_results.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_predictions(model, test_dataset, DEVICE, num_samples=8)

## Step 11: Error Analysis

Let's analyze where the model succeeds and fails to understand its limitations.

In [ ]:
@torch.no_grad()
def error_analysis(model, dataset, device, threshold=0.5):
    """Analyze per-sample F1 scores and find best/worst predictions."""
    model.eval()
    sample_scores = []
    
    for idx in tqdm(range(len(dataset)), desc='Analyzing predictions'):
        imgs, mask = dataset[idx]
        imgs_batch = imgs.unsqueeze(0).to(device)
        
        pred = torch.sigmoid(model(imgs_batch)).cpu().squeeze()
        pred_binary = (pred > threshold).float()
        mask_flat = mask.squeeze()
        
        tp = ((pred_binary == 1) & (mask_flat == 1)).sum().item()
        fp = ((pred_binary == 1) & (mask_flat == 0)).sum().item()
        fn = ((pred_binary == 0) & (mask_flat == 1)).sum().item()
        
        if tp + fp + fn > 0:
            f1 = 2 * tp / (2 * tp + fp + fn)
        else:
            f1 = 1.0  # No change and predicted no change
        
        change_area = mask_flat.sum().item() / mask_flat.numel()
        sample_scores.append({'idx': idx, 'f1': f1, 'change_area': change_area})
    
    # Sort by F1
    sample_scores.sort(key=lambda x: x['f1'])
    
    # Plot F1 distribution
    f1_values = [s['f1'] for s in sample_scores]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(f1_values, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0].axvline(np.mean(f1_values), color='red', linestyle='--', label=f'Mean F1: {np.mean(f1_values):.4f}')
    axes[0].set_xlabel('Per-sample F1 Score')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Distribution of Per-Sample F1 Scores')
    axes[0].legend()
    
    # F1 vs change area scatter
    change_areas = [s['change_area'] * 100 for s in sample_scores]
    axes[1].scatter(change_areas, f1_values, alpha=0.3, s=10, color='steelblue')
    axes[1].set_xlabel('Change Area (%)')
    axes[1].set_ylabel('F1 Score')
    axes[1].set_title('F1 Score vs. Change Area')
    
    plt.tight_layout()
    plt.savefig('error_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print worst cases
    print("\nWorst 5 predictions (with change):")
    count = 0
    for s in sample_scores:
        if s['change_area'] > 0.01:
            print(f"  Sample {s['idx']}: F1={s['f1']:.4f}, Change area={s['change_area']*100:.2f}%")
            count += 1
            if count >= 5:
                break
    
    return sample_scores

sample_scores = error_analysis(model, test_dataset, DEVICE)

## Step 12: Save Model to Hugging Face Hub (Optional)

In [ ]:
# Save final results summary
import json

results_summary = {
    'project': 'Building Change Detection using Deep Learning',
    'dataset': 'LEVIR-CD',
    'model': 'U-Net (ResNet-34 encoder)',
    'input': '6-channel (concatenated before+after RGB)',
    'image_size': IMG_SIZE,
    'epochs_trained': NUM_EPOCHS,
    'best_epoch': best_epoch,
    'test_results': test_scores,
    'training_config': {
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'optimizer': 'AdamW',
        'scheduler': 'CosineAnnealingLR',
        'loss': 'BCE + Dice',
        'augmentations': ['HorizontalFlip', 'VerticalFlip', 'RandomRotate90', 
                          'RandomBrightnessContrast', 'GaussNoise']
    }
}

with open('results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("Results saved to results_summary.json")
print(json.dumps(results_summary, indent=2))

## Step 13: Summary & Key Findings

### Results Summary

| Metric | Value |
|--------|-------|
| F1 Score | Check test_scores above |
| IoU | Check test_scores above |
| Precision | Check test_scores above |
| Recall | Check test_scores above |

### Key Observations
1. **The model learns to detect building construction/demolition** from satellite image pairs
2. **Class imbalance** (most pixels are "no change") is handled by the Dice loss component
3. **Data augmentation** (flips, rotations) helps the model generalize to different building orientations
4. **The 6-channel concatenation** approach is simple but effective for change detection

### Relevance to GIM Lab Research
- **Urban monitoring:** Automated building change detection from satellite imagery
- **Deep learning:** U-Net with pretrained encoder for geospatial semantic segmentation  
- **Remote sensing analytics:** Processing and analyzing multi-temporal satellite data
- **Practical applications:** Urban planning, disaster response, infrastructure monitoring

### Potential Extensions (Future Work)
1. Use Siamese architecture with two separate encoders sharing weights
2. Try attention mechanisms (e.g., BIT - Binary Image Transformer for Change Detection)
3. Experiment with other encoders (EfficientNet, ResNeXt)
4. Multi-class change detection (not just binary)
5. Integration with LiDAR data for 3D change detection (directly relevant to Prof. Li's research)